Notebook con las simulaciones referentes a la comparativa experimental de la librería con:
- [5] J. Luengo, S.-O. Shim, S. Alshomrani, A. Altalhi, and F. Herrera. Cnc-nos: Class noise cleaning by ensemble filtering and noise scoring. Knowledge-Based Systems, 140:27–49, 2018.

# Importo librerías

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
import warnings
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# My libraries
from filters import *
from noisers.classes import *
from noisers.funcs import *
from testFuncs import *
from noise_cv_eval import run_5cv_experiment, run_5cv_grid, run_5cv_baseline
from cleaners import CNCNOS


rs = 33

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
keel_datasets = keel_datasets = ['autos', 'car', 'cleveland', 'dermatology', 'ecoli', 'german', 'glass', 'heart', 'ionosphere', 'iris', 'lymphography',  'monk-2', 'newthyroid',    'pima',  'segment',  'sonar', 'splice', 'vehicle', 'vowel', 'wine', 'yeast'] #'magic', 'nursery', 'penbased', 'shuttle', 'thyroid',  'phoneme', 'satimage', 'ring', 'spambase', 'page-blocks', 

print(f"Available datasets: {keel_datasets}\nNumber={len(keel_datasets)}")

Available datasets: ['autos', 'car', 'cleveland', 'dermatology', 'ecoli', 'german', 'glass', 'heart', 'ionosphere', 'iris', 'lymphography', 'monk-2', 'newthyroid', 'pima', 'segment', 'sonar', 'splice', 'vehicle', 'vowel', 'wine', 'yeast']
Number=21


# Class random noise

Configuro los filtros a probar:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import FunctionTransformer

# drop_missing = FunctionTransformer(
#     lambda X: X.dropna(),
#     validate=False
# )

classifiers = [
    KNeighborsClassifier(
        n_neighbors=1
        ),
    SVC(
        C=100, 
        kernel="rbf",
        gamma=0.3,
        tol=1e-12
        ),
    # DecisionTreeClassifier(
    #     criterion="entropy",
    #     min_samples_leaf=2,
    #     random_state=42
    # )
]
clf_names = [
    "knn", 
    "svm", 
    #"c45"
    ]

# Create the distance_based_filter_list
filter_grid = [
    # Basados en distancia
    {
        "name": f"ENN",
        "filter": ENNFilter(n_neighbors=3, mode="enn", n_jobs=-1),
    },
    {
        "name": f"ENNTh",
        "filter": ENNProbFilter(n_neighbors=3, mode="th", threshold=0.7, n_jobs=-1),
    },
    {
        "name": f"NCNEdit",
        "filter": NCNEdit(n_neighbors=3, n_jobs=-1),
    },

    # Basados en clasificadores
    {
        "name": f"CF",
        "filter": ClassificationFilter(estimator=KNeighborsClassifier(1), cv=10),
    },
    # {
    #     "name": f"CVCF",
    #     "filter": CVCFFilter(vote_rule="consensus", cv=4),
    # },
    # {
    #     "name": f"EF",
    #     "filter": EnsembleFiltering(vote_rule="consensus", cv=4),
    # },
    # {
    #     "name": f"IPF",
    #     "filter": IterativePartitioningFilter(vote_rule="majority", cv=5, p_stop=0.01, ),
    # },
    # {
    #     "name": f"INFFC_old_wrong",
    #     "filter": INFFC_old_wrong(vote_rule="consensus", cv=3),
    # },

    # # Filtros de limpieza
    # {
    #     "name": f"CNC-NOS",
    #     "filter": CNCNOS(),
    # },
]

# Specify dataset info
noise_type = "cla_rand"
folds = (1, 2, 3, 4, 5)



for noise_k in [5]:
    for seed in range(1,6):
        for clf_idx, classifier in enumerate(classifiers):
            all_results = []
            for ds_idx, dataset in enumerate(keel_datasets):

                print(f"noise pct: {noise_k} - seed0{seed} - classifier {clf_names[clf_idx]} - {ds_idx},{dataset}")
                
                # Compute baseline (no filter) results
                baseline_df = run_5cv_baseline(
                    dataset=dataset,
                    noise_type=noise_type,
                    seed=seed,
                    classifier = classifier,
                    k=noise_k,
                    test_source="noisy",
                    folds=folds,
                )

                # Compute results using filters
                distance_experiments_5 = [
                    {
                        "dataset": dataset,
                        "noise_type": noise_type,
                        "seed": seed,
                        "k": noise_k,
                        "filters": {cfg["name"]: cfg["filter"]},
                        "test_source": "clean",
                        "folds": folds,
                        "summarize": True,
                        "classifier": classifier,
                        "experiment_name": f"{dataset}|{noise_type}|{noise_k}|{cfg['name']}",   
                    }
                    for cfg in filter_grid
                ]
                classification_df, removal_df, class_summary_df, removal_summary_df = run_5cv_grid(
                    distance_experiments_5,
                    save_path="results_5cv",
                    save_params = False,
                    save_format="pickle",
                    warnings_path="results_5cv/warnings_run_5cv_grid____.txt",
                    clear_warnings_file=False,)

                # def drop_params(df):
                #     return df.drop(columns=["params"], errors="ignore")

                all_results.append({
                    "dataset": dataset,
                    "baseline_df": baseline_df,
                    "classification_df": classification_df, #drop_params(classification_df),
                    "removal_df": removal_df, #drop_params(removal_df),
                    "class_summary_df":class_summary_df, #drop_params(class_summary_df),
                    "removal_summary_df": removal_summary_df,#drop_params(removal_summary_df),
                })
                with open(f"./resultsEvaluation/{noise_k}/seed0{seed}/{clf_names[clf_idx]}/dists_res.pkl", "wb") as f:
                    pickle.dump(all_results,f)

noise pct: 5 - seed01 - classifier knn - 0,autos


5CV experiments: 100%|██████████| 4/4 [00:01<00:00,  2.71it/s]


noise pct: 5 - seed01 - classifier knn - 1,car


5CV experiments: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]


noise pct: 5 - seed01 - classifier knn - 2,cleveland


...

5CV experiments: 100%|██████████| 4/4 [00:05<00:00,  1.29s/it]


noise pct: 5 - seed05 - classifier svm - 19,wine


5CV experiments: 100%|██████████| 4/4 [00:01<00:00,  3.36it/s]


noise pct: 5 - seed05 - classifier svm - 20,yeast


5CV experiments: 100%|██████████| 4/4 [00:11<00:00,  2.82s/it]
